# Notebook 02 — Baseline: Naive vs Structured Prompts

**Phase 3 | Ablation Cells A & C (no ControlNet)**

This notebook:
1. Picks 10 curated (breed, condition, environment) triples from the taxonomy.
2. Generates 4 images per triple for both **naive** (Cell A) and **structured** (Cell C) prompts.
3. Saves every image with a JSON metadata sidecar to `outputs/A/` and `outputs/C/`.
4. Renders a qualitative comparison grid (naive left, structured right) for visual inspection.

> **Run on Colab (T4/V100/A100).** Mac M2 is for editing only.

## 0 — Colab Bootstrap

Mount Google Drive, clone the repo (or pull if it already exists), and install dependencies.

In [ ]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_URL  = 'https://github.com/<YOUR_USERNAME>/stable-diffusion.git'  # ← update
    REPO_DIR  = '/content/stable-diffusion'

    if not os.path.isdir(REPO_DIR):
        !git clone $REPO_URL $REPO_DIR
    else:
        !git -C $REPO_DIR pull

    %cd $REPO_DIR
    !pip install -q -r requirements.txt
else:
    # Local: ensure we're at repo root
    root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if os.path.basename(root) == 'stable-diffusion':
        os.chdir(root)
    print(f'Working dir: {os.getcwd()}')

## 1 — Verify GPU & Imports

In [ ]:
import torch

device = (
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f'Device : {device}')

if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from tqdm.auto import tqdm

from src.data.taxonomy import load_taxonomy, all_breeds, all_conditions, all_environments, species_for_breed
from src.generation.seeds import get_seeds
from src.generation.io import save_image_with_sidecar
from src.pipelines.baseline import run_baseline
from src.prompts.mapper import structured_input_to_prompt, naive_input_to_prompt

print('All imports OK')

## 2 — Define the 10 Curated Triples

These triples are hand-picked for visual variety: mix of species, conditions, and environments.
They are reused across all ablation cells so comparisons are apples-to-apples.

In [ ]:
# Each entry: (breed, condition_key, environment_key)
TRIPLES = [
    ('beagle',              'cone_collar',    'clinic'),
    ('Siamese',             'health_exam',    'exam_room'),
    ('golden_retriever',    'bandaged_paw',   'home'),          # NOTE: golden_retriever not in Oxford-37; will be "golden retriever" in prompt
    ('Persian',             'grooming',       'grooming_salon'),
    ('pug',                 'weight_check',   'clinic'),
    ('Maine_Coon',          'dental_check',   'exam_room'),
    ('samoyed',             'post_bath_drying', 'home'),
    ('British_Shorthair',   'vaccination',    'mobile_clinic'),
    ('boxer',               'cone_collar',    'outdoor'),
    ('Ragdoll',             'health_exam',    'home'),
]

# Validate all condition/environment keys exist in taxonomy
tax = load_taxonomy()
conds = all_conditions(tax)
envs  = all_environments(tax)

for breed, cond, env in TRIPLES:
    assert cond in conds, f'Unknown condition: {cond}'
    assert env  in envs,  f'Unknown environment: {env}'

print(f'{len(TRIPLES)} triples validated ✓')
for i, (b, c, e) in enumerate(TRIPLES, 1):
    sp = species_for_breed(b)
    print(f'  {i:2d}. {b} ({sp}) | {c} | {e}')

## 3 — Preview Prompts (no generation yet)

Inspect what the naive vs structured prompts look like for each triple before spending GPU time.

In [ ]:
for breed, cond, env in TRIPLES:
    sp = species_for_breed(breed)

    naive_pp = naive_input_to_prompt(breed=breed, condition=cond)
    struct_pp = structured_input_to_prompt(
        breed=breed, species=sp, condition=cond, environment=env
    )

    print('─' * 80)
    print(f'Breed: {breed}  |  Condition: {cond}  |  Env: {env}')
    print(f'  [NAIVE]      {naive_pp.positive}')
    print(f'  [STRUCTURED] {struct_pp.positive}')

## 4 — Load Baseline Pipeline

The pipeline is loaded **once** (cached via `lru_cache`). This takes ~30–60s on Colab depending on network / Drive cache.

In [ ]:
from src.pipelines.loader import load_baseline_pipeline

pipe = load_baseline_pipeline()  # cached — safe to call from baseline.py simultaneously
print(f'Pipeline loaded on {next(pipe.unet.parameters()).device}')

## 5 — Generate: Cell A (Naive + No ControlNet) & Cell C (Structured + No ControlNet)

**10 triples × 4 seeds × 2 cells = 80 images total.**

On a T4 (~30 steps, 512×512) each image takes ~5–8s → total ≈ 10–12 min.

In [ ]:
SEEDS = get_seeds()           # [42, 137, 2024, 9999] from generation.yaml
OUT_ROOT = Path('outputs')

results: dict[str, list[dict]] = {'A': [], 'C': []}  # cell → list of {image, meta, stem}

total = len(TRIPLES) * len(SEEDS) * 2
pbar  = tqdm(total=total, desc='Generating')

for breed, cond, env in TRIPLES:
    sp = species_for_breed(breed)

    naive_pp  = naive_input_to_prompt(breed=breed, condition=cond)
    struct_pp = structured_input_to_prompt(
        breed=breed, species=sp, condition=cond, environment=env
    )

    for seed in SEEDS:
        breed_slug = breed.lower().replace(' ', '_')
        stem_base  = f'{breed_slug}__{cond}__{env}__{seed}'

        # ── Cell A: naive, no ControlNet ────────────────────────────────────
        img_a, meta_a = run_baseline(
            naive_pp,
            seed=seed, breed=breed, species=sp,
            condition=cond, environment=env, cell='A',
        )
        img_path_a, _ = save_image_with_sidecar(
            img_a, OUT_ROOT / 'A', f'A__{stem_base}', meta_a
        )
        results['A'].append({'image': img_a, 'meta': meta_a, 'stem': stem_base})
        pbar.update(1)

        # ── Cell C: structured, no ControlNet ───────────────────────────────
        img_c, meta_c = run_baseline(
            struct_pp,
            seed=seed, breed=breed, species=sp,
            condition=cond, environment=env, cell='C',
        )
        img_path_c, _ = save_image_with_sidecar(
            img_c, OUT_ROOT / 'C', f'C__{stem_base}', meta_c
        )
        results['C'].append({'image': img_c, 'meta': meta_c, 'stem': stem_base})
        pbar.update(1)

pbar.close()
n_a = len(list((OUT_ROOT / 'A').glob('*.png')))
n_c = len(list((OUT_ROOT / 'C').glob('*.png')))
print(f'Cell A: {n_a} images  |  Cell C: {n_c} images')

## 6 — Qualitative Comparison Grid

Each row = one (breed, condition, environment) triple.  
Left 4 columns = Cell A (naive) | Right 4 columns = Cell C (structured)  
Columns within each half = the 4 seeds.

In [ ]:
import matplotlib.patches as mpatches

N_TRIPLES = len(TRIPLES)
N_SEEDS   = len(SEEDS)
N_COLS    = N_SEEDS * 2  # 8 columns: 4 naive + 4 structured

FIG_W = 3.0 * N_COLS
FIG_H = 3.2 * N_TRIPLES

fig, axes = plt.subplots(N_TRIPLES, N_COLS, figsize=(FIG_W, FIG_H))
fig.patch.set_facecolor('#0d0d0d')

# Column header row shading colours
NAIVE_CLR      = '#1a3050'
STRUCTURED_CLR = '#1a4030'

for row_idx, (breed, cond, env) in enumerate(TRIPLES):
    breed_slug = breed.lower().replace(' ', '_')
    stem_base  = f'{breed_slug}__{cond}__{env}'

    for col_idx, seed in enumerate(SEEDS):
        # ── Naive (left block) ───────────────────────────────────────────────
        ax_a = axes[row_idx][col_idx]
        img_a = results['A'][row_idx * N_SEEDS + col_idx]['image']
        ax_a.imshow(img_a)
        ax_a.axis('off')
        ax_a.set_facecolor(NAIVE_CLR)
        if row_idx == 0:
            ax_a.set_title(f'Naive\nseed={seed}', color='#aac8e0', fontsize=8, pad=4)

        # ── Structured (right block) ─────────────────────────────────────────
        ax_c = axes[row_idx][col_idx + N_SEEDS]
        img_c = results['C'][row_idx * N_SEEDS + col_idx]['image']
        ax_c.imshow(img_c)
        ax_c.axis('off')
        ax_c.set_facecolor(STRUCTURED_CLR)
        if row_idx == 0:
            ax_c.set_title(f'Structured\nseed={seed}', color='#aae0c8', fontsize=8, pad=4)

    # Row label (leftmost axis)
    sp = species_for_breed(breed)
    label = f'{breed.replace("_", " ")}\n({sp})\n{cond}\n{env}'
    axes[row_idx][0].set_ylabel(label, color='white', fontsize=7,
                                rotation=0, ha='right', va='center', labelpad=60)

# Legend patches
patch_a = mpatches.Patch(color=NAIVE_CLR,      label='Cell A — naive prompt')
patch_c = mpatches.Patch(color=STRUCTURED_CLR, label='Cell C — structured prompt')
fig.legend(handles=[patch_a, patch_c], loc='upper center', ncol=2,
           fontsize=10, facecolor='#1a1a1a', edgecolor='gray', labelcolor='white',
           bbox_to_anchor=(0.5, 1.01))

fig.suptitle('Baseline SD 1.5 — Naive vs Structured Prompts (Cells A & C)',
             color='white', fontsize=13, y=1.03)

plt.tight_layout(pad=0.5)
grid_path = OUT_ROOT / 'phase3_comparison_grid.png'
plt.savefig(grid_path, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'Grid saved → {grid_path}')

## 7 — Quick Sanity Check: Metadata Sidecar

Verify that each image has a matching `.json` sidecar and that the fields are populated correctly.

In [ ]:
import json

for cell in ('A', 'C'):
    jsons = sorted((OUT_ROOT / cell).glob('*.json'))
    print(f'Cell {cell}: {len(jsons)} sidecars')
    if jsons:
        # Show the first sidecar as a spot-check
        sample = json.loads(jsons[0].read_text())
        print(f'  Sample ({jsons[0].name}):')
        for k, v in sample.items():
            val_str = str(v)[:80] + ('…' if len(str(v)) > 80 else '')
            print(f'    {k:30s} {val_str}')
    print()

## 8 — Single-Seed Close-Up Comparison

For seed=42 only, show a cleaner 2-column grid (naive | structured) for easier visual inspection.

In [ ]:
SEED_IDX = 0  # index of seed=42 in SEEDS list

fig2, axes2 = plt.subplots(N_TRIPLES, 2, figsize=(8, 3.2 * N_TRIPLES))
fig2.patch.set_facecolor('#0d0d0d')

for i, (breed, cond, env) in enumerate(TRIPLES):
    sp = species_for_breed(breed)
    label = f'{breed.replace("_"," ")} | {cond} | {env}'

    img_a = results['A'][i * N_SEEDS + SEED_IDX]['image']
    img_c = results['C'][i * N_SEEDS + SEED_IDX]['image']

    axes2[i][0].imshow(img_a)
    axes2[i][0].axis('off')
    axes2[i][0].set_title(f'Naive — {label}', color='#aac8e0', fontsize=7)

    axes2[i][1].imshow(img_c)
    axes2[i][1].axis('off')
    axes2[i][1].set_title(f'Structured — {label}', color='#aae0c8', fontsize=7)

fig2.suptitle(f'Seed={SEEDS[SEED_IDX]} close-up: Naive (A) vs Structured (C)',
              color='white', fontsize=11)
plt.tight_layout(pad=0.5)
close_up_path = OUT_ROOT / f'phase3_seed{SEEDS[SEED_IDX]}_closeup.png'
plt.savefig(close_up_path, dpi=150, bbox_inches='tight', facecolor=fig2.get_facecolor())
plt.show()
print(f'Close-up saved → {close_up_path}')

## 9 — Observations Template

Fill this in after reviewing the grids above. These notes feed directly into the Phase 5 evaluation report and the slide deck.

---

### Visual Observations (Cell A vs Cell C, seed=42)

| Triple | Cell A (Naive) | Cell C (Structured) | Winner |
|--------|---------------|--------------------|---------|
| beagle / cone_collar / clinic | _describe_ | _describe_ | |
| Siamese / health_exam / exam_room | _describe_ | _describe_ | |
| golden_retriever / bandaged_paw / home | _describe_ | _describe_ | |
| Persian / grooming / grooming_salon | _describe_ | _describe_ | |
| pug / weight_check / clinic | _describe_ | _describe_ | |
| Maine_Coon / dental_check / exam_room | _describe_ | _describe_ | |
| samoyed / post_bath_drying / home | _describe_ | _describe_ | |
| British_Shorthair / vaccination / mobile_clinic | _describe_ | _describe_ | |
| boxer / cone_collar / outdoor | _describe_ | _describe_ | |
| Ragdoll / health_exam / home | _describe_ | _describe_ | |

### Failure cases to investigate in Phase 5 evaluation
- _List any images where anatomy looked deformed, condition was missed, environment was wrong, etc._

### Notes for Phase 4 (ControlNet)
- _Any observations about how shape/pose control might help specific triples_

---
**End of Notebook 02** — Phase 3 complete.  
Next → `03_controlnet_generation.ipynb` (Phase 4).